# Assignment 2, notebook 4: Explaining the model and turning it into decisions (Part C6 and business use)

Uses the final model from notebook 3 (LightGBM, selected features).

1. **Key risk drivers:** which features push risk up or down, using SHAP values (each feature's contribution to each loan's score)
2. **Plain-English driver table:** actual bad-loan rates behind each top driver
3. **Explaining single loans:** why one loan scores high and another low
4. **Recalibration:** making predicted probabilities match reality
5. **Risk tiers:** five bands with actual bad-loan rates and dollars lost
6. **Cost-based review rule:** which loans to review, given what a review costs and what it saves
7. **Pricing view:** expected loss by tier

In [1]:
import json, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
import lightgbm as lgb
warnings.filterwarnings("ignore")

In [2]:
TEAL, GREY, RED, AMBER = "#0E7C86", "#B8BFCC", "#C0392B", "#D98E04"
PROC = Path("../../data/processed"); OUT = Path("../outputs")
tr, va, te = (pd.read_parquet(PROC / f"a2_{s}.parquet") for s in ["train", "valid", "test"])
cfg = json.load(open(OUT / "a2_final_model_config.json"))
SEL = cfg["selected_features"]
booster = lgb.Booster(model_file=str(OUT / "a2_final_model.txt"))
p_va, p_te = booster.predict(va[SEL]), booster.predict(te[SEL])
print(f"{len(SEL)} features | test AUC {roc_auc_score(te['bad'], p_te):.4f} (should match notebook 3)")

30 features | test AUC 0.7521 (should match notebook 3)


## 1. Key risk drivers (SHAP)
*Why:* permutation importance (notebook 3) says *how much* a feature matters; SHAP also says *which way* it pushes each loan's risk. LightGBM computes SHAP values directly (`pred_contrib=True`), so no extra package is needed. Values are on the log-odds scale: positive = raises risk, negative = lowers it.

In [3]:
sample = te.sample(15000, random_state=0)
contrib = booster.predict(sample[SEL], pred_contrib=True)
shap_df = pd.DataFrame(contrib[:, :-1], columns=SEL, index=sample.index)   # last column is the baseline

imp = shap_df.abs().mean().sort_values(ascending=False)
corr = {f: np.corrcoef(sample[f], shap_df[f])[0, 1] if sample[f].nunique() > 1 else 0 for f in imp.index}
top = pd.DataFrame({"mean |SHAP|": imp, "share of total": imp / imp.sum(),
                    "direction": ["higher value -> more risk" if corr[f] > 0 else "higher value -> less risk" for f in imp.index]})
display(top.head(15).round(4))
px.bar(top.head(15).iloc[::-1].reset_index(), x="mean |SHAP|", y="index", orientation="h", color="direction",
       color_discrete_map={"higher value -> more risk": RED, "higher value -> less risk": TEAL}, template="plotly_white",
       title="Top 15 risk drivers (average size of SHAP contribution, FY2019 sample)", labels={"index": ""}).show()

,mean |SHAP|,share of total,direction
rate_vs_year,0.4299,0.2258,higher value -> more risk
lender_te,0.3198,0.1680,higher value -> more risk
naics3_te,0.1889,0.0992,higher value -> more risk
variable_rate,0.1025,0.0538,higher value -> more risk
biz_age_Existing 2y+,0.1020,0.0536,higher value -> less risk
lender_log_volume,0.0973,0.0511,higher value -> more risk
log_amount,0.0845,0.0444,higher value -> less risk
amount_per_job_log,0.0679,0.0357,higher value -> less risk
jobs_log,0.0549,0.0288,higher value -> more risk
out_of_state_lender,0.0533,0.0280,higher value -> more risk


### How the top numeric drivers change risk
Each dot is a loan: the feature's value (x) against its contribution to risk (y). Above zero raises risk.

In [4]:
NAMES = {"rate_vs_year": "Interest rate vs that year's median (points)", "lender_te": "Lender's historical bad-loan rate",
         "naics3_te": "Industry's historical bad-loan rate", "log_amount": "Loan amount (log10 $)",
         "lender_log_volume": "Lender size (log10 loans)", "amount_per_job_log": "Loan amount per job (log10 $)",
         "guarantee_pct": "SBA guarantee share", "jobs_log": "Jobs supported (log)"}
num_top = [f for f in imp.index if sample[f].nunique() > 10][:6]
from plotly.subplots import make_subplots
fig = make_subplots(rows=2, cols=3, subplot_titles=[NAMES.get(f, f) for f in num_top])
for i, f in enumerate(num_top):
    s = sample[[f]].assign(shap=shap_df[f]).sample(4000, random_state=1)
    fig.add_trace(go.Scattergl(x=s[f], y=s["shap"], mode="markers", marker=dict(size=3, color=TEAL, opacity=0.35), showlegend=False),
                  row=i // 3 + 1, col=i % 3 + 1)
fig.add_hline(y=0, line_dash="dot", line_color=GREY)
fig.update_layout(height=600, template="plotly_white", title="SHAP dependence: how each top driver moves risk")
fig.show()

## 2. Plain-English driver table
*Why:* SHAP values are abstract. This shows the actual FY2019 bad-loan rate in the lowest and highest fifth of each top numeric driver, and for each top yes/no feature.

In [5]:
rows = []
for f in num_top:
    q = pd.qcut(te[f].rank(method="first"), 5, labels=False)
    g = te.groupby(q)["bad"].mean()
    rows.append({"driver": NAMES.get(f, f), "bad rate, lowest 20%": g.iloc[0], "bad rate, highest 20%": g.iloc[-1],
                 "ratio high/low": g.iloc[-1] / g.iloc[0]})
for f in [x for x in imp.index if te[x].nunique() == 2][:8]:
    g = te.groupby(te[f])["bad"].mean()
    rows.append({"driver": f, "bad rate, lowest 20%": g.get(0, np.nan), "bad rate, highest 20%": g.get(1, np.nan),
                 "ratio high/low": g.get(1, np.nan) / g.get(0, np.nan)})
drivers = pd.DataFrame(rows).set_index("driver")
drivers.columns = ["bad rate: low value / flag = 0", "bad rate: high value / flag = 1", "ratio"]
drivers.round(3)

,bad rate: low value / flag = 0,bad rate: high value / flag = 1,ratio
driver,,,
Interest rate vs that year's median (points),0.018,0.126,7.080
Lender's historical bad-loan rate,0.025,0.137,5.429
Industry's historical bad-loan rate,0.034,0.100,2.984
Lender size (log10 loans),0.033,0.069,2.059
Loan amount (log10 $),0.102,0.019,0.185
Loan amount per job (log10 $),0.078,0.031,0.395
variable_rate,0.040,0.073,1.800
biz_age_Existing 2y+,0.077,0.060,0.776
out_of_state_lender,0.040,0.079,1.976


## 3. Explaining single loans
*Why:* a loan officer needs to know *why* a loan scored high. We show the five biggest contributions for a high-, middle- and low-risk FY2019 loan.

In [6]:
sample = sample.assign(p=booster.predict(sample[SEL]))
picks = {"High risk": sample["p"].idxmax(), "Middle": (sample["p"] - sample["p"].median()).abs().idxmin(), "Low risk": sample["p"].idxmin()}
fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{k}: predicted {sample.loc[i, 'p']:.1%}" for k, i in picks.items()])
for j, (k, i) in enumerate(picks.items()):
    s = shap_df.loc[i]; s = s.reindex(s.abs().sort_values(ascending=False).index[:5])[::-1]
    lab = [f"{NAMES.get(f, f)[:28]} = {sample.loc[i, f]:.2f}" for f in s.index]
    fig.add_trace(go.Bar(x=s.values, y=lab, orientation="h", marker_color=[RED if v > 0 else TEAL for v in s.values], showlegend=False),
                  row=1, col=j + 1)
fig.update_layout(height=380, template="plotly_white", title="Top five reasons behind three loans' scores (red raises risk, teal lowers it)")
fig.show()

## 4. Recalibration
*Why:* notebook 3 showed the model's probabilities run high on FY2019 (the riskiest tenth: 28% predicted vs 21% actual). Pricing and expected-loss figures need probabilities that match reality.
We fit an **isotonic regression** (a flexible, order-preserving mapping from score to probability) on FY2018 and apply it to FY2019. It cannot change the ranking, so AUC is unchanged.

In [7]:
iso = IsotonicRegression(out_of_bounds="clip").fit(p_va, va["bad"])
q_te = iso.predict(p_te)
dec = pd.qcut(pd.Series(p_te).rank(method="first"), 10, labels=range(1, 11)).astype(int).values
cal = pd.DataFrame({"decile": dec, "raw": p_te, "recalibrated": q_te, "actual": te["bad"].values}).groupby("decile").mean()
display(cal.sort_index(ascending=False).round(3))
print(f"Brier score: raw {brier_score_loss(te['bad'], p_te):.4f} -> recalibrated {brier_score_loss(te['bad'], q_te):.4f}"
      f" | mean predicted: raw {p_te.mean():.3f}, recalibrated {q_te.mean():.3f}, actual {te['bad'].mean():.3f}"
      f" | AUC {roc_auc_score(te['bad'], q_te):.4f}")
c = cal.reset_index()
fig = go.Figure([go.Scatter(x=c["actual"] * 100, y=c["raw"] * 100, mode="markers+lines", name="Raw model", line=dict(color=RED)),
                 go.Scatter(x=c["actual"] * 100, y=c["recalibrated"] * 100, mode="markers+lines", name="Recalibrated", line=dict(color=TEAL)),
                 go.Scatter(x=[0, 25], y=[0, 25], mode="lines", name="Perfect", line=dict(color=GREY, dash="dot"))])
fig.update_layout(template="plotly_white", title="Calibration by risk decile, FY2019", xaxis_title="Actual bad-loan rate (%)",
                  yaxis_title="Predicted (%)", height=420); fig.show()

,raw,recalibrated,actual
decile,,,
10,0.279,0.264,0.214
9,0.153,0.135,0.121
8,0.111,0.113,0.086
7,0.084,0.084,0.066
6,0.065,0.063,0.054
5,0.050,0.047,0.043
4,0.038,0.036,0.030
3,0.027,0.023,0.024
2,0.018,0.016,0.015


Brier score: raw 0.0581 -> recalibrated 0.0579 | mean predicted: raw 0.083, recalibrated 0.079, actual 0.066 | AUC 0.7515


**Note:** some over-prediction can remain, because FY2019 loans are younger than FY2018 loans and fewer have had time to fail. In use, the calibration should be refreshed each year on the most recent fully-observed loans.

## 5. Risk tiers
*Why:* a lending team works with a few bands, not a probability. Tier boundaries are set on FY2018 scores (so the test year is untouched): A = safest 40%, B = next 30%, C = next 15%, D = next 10%, E = riskiest 5%.

In [8]:
cuts = np.quantile(p_va, [0.40, 0.70, 0.85, 0.95])
TIERS = ["A (safest)", "B", "C", "D", "E (riskiest)"]
def tier(p): return pd.Categorical(np.array(TIERS)[np.searchsorted(cuts, p, side="right")], categories=TIERS, ordered=True)

t = te.assign(tier=tier(p_te), q=q_te, lost=te["grosschargeoffamount"].fillna(0),
              lender_loss=te["grosschargeoffamount"].fillna(0) * (1 - te["guarantee_pct"]))
tiers = t.groupby("tier", observed=False).agg(loans=("bad", "size"), predicted=("q", "mean"), actual_bad_rate=("bad", "mean"),
                                              approved_usd_m=("grossapproval", lambda s: s.sum() / 1e6),
                                              lost_usd_m=("lost", lambda s: s.sum() / 1e6))
tiers["share of loans"] = tiers["loans"] / tiers["loans"].sum()
tiers["share of bad loans"] = t.groupby("tier", observed=False)["bad"].sum() / t["bad"].sum()
tiers["share of $ lost"] = tiers["lost_usd_m"] / tiers["lost_usd_m"].sum()
tiers["loss rate on $ approved"] = tiers["lost_usd_m"] / tiers["approved_usd_m"]
display(tiers.round(3))
fig = go.Figure([go.Bar(x=TIERS, y=tiers["actual_bad_rate"] * 100, marker_color=[TEAL, TEAL, AMBER, RED, RED],
                        text=(tiers["actual_bad_rate"] * 100).round(1), textposition="outside")])
fig.add_hline(y=te["bad"].mean() * 100, line_dash="dot", line_color=GREY, annotation_text=f"Overall {te['bad'].mean():.1%}")
fig.update_layout(template="plotly_white", title="Actual bad-loan rate by risk tier, FY2019", yaxis_title="Bad-loan rate (%)"); fig.show()

,loans,predicted,actual_bad_rate,approved_usd_m,lost_usd_m,share of loans,share of bad loans,share of $ lost,loss rate on $ approved
tier,,,,,,,,,
A (safest),19220,0.021,0.020,13487.819,104.552,0.421,0.128,0.247,0.008
B,13553,0.069,0.056,4765.975,131.009,0.297,0.253,0.309,0.027
C,6191,0.119,0.099,1369.767,70.997,0.136,0.203,0.168,0.052
D,4201,0.173,0.147,614.881,53.360,0.092,0.205,0.126,0.087
E (riskiest),2492,0.319,0.255,312.470,63.420,0.055,0.212,0.150,0.203


## 6. Cost-based review rule
*Why:* "how many loans should we review?" depends on money, not on F1. Ranking by probability alone treats a $20k and a $2m loan the same, so we rank by **expected lender loss** instead and review a loan only when the expected saving beats the review cost.

For each loan: **expected saving = recalibrated probability of going bad x lender exposure x share of loss a review avoids**, where lender exposure = loan amount x (1 - SBA guarantee share) x the share of that exposure typically lost on bad loans (measured on FY2018).

Assumptions (illustrative, a lending team would use its own): review cost **$500**; a review avoids **20%** of the loss on a loan that would go bad (better structuring, collateral, early monitoring). The rule and its loss-severity figure are set on FY2018 and evaluated on FY2019.

In [9]:
q_va = iso.predict(p_va)
def lender_exposure(D): return D["grossapproval"] * (1 - D["guarantee_pct"])
def lender_loss(D): return D["grosschargeoffamount"].fillna(0) * (1 - D["guarantee_pct"])
sev = lender_loss(va)[va.bad == 1].sum() / lender_exposure(va)[va.bad == 1].sum()   # share of exposure lost when a loan goes bad (FY2018)
print(f"On FY2018 bad loans the lender lost {sev:.0%} of its (unguaranteed) exposure")

def evaluate(D, q, rule_flag, name):
    loss = lender_loss(D).values; n = rule_flag.sum()
    return {"policy": name, "loans reviewed %": rule_flag.mean() * 100, "bad loans caught %": D["bad"].values[rule_flag].sum() / D["bad"].sum() * 100,
            "lender loss covered %": loss[rule_flag].sum() / loss.sum() * 100,
            "net benefit $m": (SAVE * loss[rule_flag].sum() - COST * n) / 1e6}

COST, SAVE = 500, 0.20
exp_save_te = q_te * lender_exposure(te).values * sev * SAVE
rule = exp_save_te > COST
order_p = np.argsort(-p_te); same_n = np.zeros(len(te), bool); same_n[order_p[: rule.sum()]] = True
rnd = np.zeros(len(te), bool); rnd[np.random.default_rng(0).permutation(len(te))[: rule.sum()]] = True
policies = pd.DataFrame([evaluate(te, q_te, rule, "Expected-saving rule (review if saving > cost)"),
                         evaluate(te, q_te, same_n, "Same number of loans, ranked by probability only"),
                         evaluate(te, q_te, rnd, "Same number of loans, picked at random")]).set_index("policy")
display(policies.round(2))

sens = []
for cost in [250, 500, 1000, 2000]:
    for save in [0.10, 0.20, 0.30]:
        COST, SAVE = cost, save
        r = q_te * lender_exposure(te).values * sev * SAVE > COST
        sens.append({"review cost $": cost, "share of loss a review avoids": f"{save:.0%}", **{k: v for k, v in evaluate(te, q_te, r, "").items() if k != "policy"}})
COST, SAVE = 500, 0.20
sens = pd.DataFrame(sens)
print("Sensitivity: share of FY2019 loans the rule would review, and the net benefit")
display(sens.pivot(index="review cost $", columns="share of loss a review avoids", values="loans reviewed %").round(1))
display(sens.pivot(index="review cost $", columns="share of loss a review avoids", values="net benefit $m").round(2))

On FY2018 bad loans the lender lost 71% of its (unguaranteed) exposure


,loans reviewed %,bad loans caught %,lender loss covered %,net benefit $m
policy,,,,
Expected-saving rule (review if saving > cost),38.99,46.96,75.70,8.84
"Same number of loans, ranked by probability only",38.99,72.96,55.87,4.19
"Same number of loans, picked at random",38.99,39.11,40.86,0.67


Sensitivity: share of FY2019 loans the rule would review, and the net benefit


share of loss a review avoids,10%,20%,30%
review cost $,,,
250,39.0,59.6,70.3
500,20.6,39.0,51.4
1000,8.1,20.6,30.9
2000,2.3,8.1,14.4


share of loss a review avoids,10%,20%,30%
review cost $,,,
250,4.42,14.50,25.28
500,1.48,8.84,18.75
1000,-0.27,2.96,9.70
2000,-0.75,-0.55,1.47


## 7. Pricing view: expected loss by tier
*Why:* the brief asks about pricing by risk. A rough annual risk premium per tier = recalibrated probability of a bad loan x the share of the approved amount the lender loses on a bad loan (after the SBA guarantee, measured on FY2018), spread over an assumed 7-year average loan life. It is a starting point for pricing, not a full pricing model (no funding cost, servicing or capital).

In [10]:
sev_p = lender_loss(va)[va.bad == 1].sum() / va.loc[va.bad == 1, "grossapproval"].sum()   # measured on FY2018, not the test year
LIFE_YEARS = 7
price = pd.DataFrame({"recalibrated probability of bad loan": tiers["predicted"]})
price["expected lifetime lender loss (% of loan)"] = price["recalibrated probability of bad loan"] * sev_p
price["approx. annual risk premium (bp)"] = price["expected lifetime lender loss (% of loan)"] / LIFE_YEARS * 1e4
price["premium vs tier A (bp)"] = price["approx. annual risk premium (bp)"] - price["approx. annual risk premium (bp)"].iloc[0]
display(price.round(4))
print(f"Lender loss on bad loans = {sev_p:.1%} of the approved amount after the SBA guarantee (FY2018); spread over {LIFE_YEARS} years.")

,recalibrated probability of bad loan,expected lifetime lender loss (% of loan),approx. annual risk premium (bp),premium vs tier A (bp)
tier,,,,
A (safest),0.0212,0.0040,5.7368,0.0000
B,0.0686,0.0130,18.5960,12.8592
C,0.1189,0.0226,32.2276,26.4908
D,0.1730,0.0328,46.9006,41.1638
E (riskiest),0.3187,0.0605,86.3997,80.6629


Lender loss on bad loans = 19.0% of the approved amount after the SBA guarantee (FY2018); spread over 7 years.


### What each tier means for the lending team

| Tier | Share of loans | Suggested action |
|---|---|---|
| A (safest) | 40% | Standard, fast-track approval; lightest monitoring |
| B | 30% | Standard process |
| C | 15% | Standard process, closer annual monitoring |
| D | 10% | Senior underwriter review; consider collateral or a higher rate; quarterly monitoring |
| E (riskiest) | 5% | Full manual review before approval; price for risk or restructure; early-warning monitoring and proactive collections |

In [11]:
tiers.to_csv(OUT / "a2_risk_tiers.csv"); policies.to_csv(OUT / "a2_review_policies.csv"); sens.to_csv(OUT / "a2_review_sensitivity.csv", index=False)
top.to_csv(OUT / "a2_shap_importance.csv"); drivers.to_csv(OUT / "a2_driver_table.csv"); price.to_csv(OUT / "a2_pricing_view.csv")
json.dump({"tier_cuts_raw_score": cuts.tolist(), "tiers": TIERS}, open(OUT / "a2_tier_config.json", "w"), indent=1)
print("saved")

saved


## Summary

**Key risk drivers** (SHAP, confirmed by actual FY2019 bad-loan rates)
- **Interest rate relative to that year's market** is the strongest driver (23% of total SHAP). Loans priced in the top fifth go bad 12.6% of the time vs 1.8% in the bottom fifth (7 times). Lenders already price for risk they can see; the model reads that signal.
- **Lender's track record** (17%): loans from the worst-performing fifth of lenders go bad 13.7% of the time vs 2.5% (5 times).
- **Industry's track record** (10%): 10.0% vs 3.4% (3 times).
- **Loan size:** small loans are riskier, 10.2% in the smallest fifth vs 1.9% in the largest.
- **Loan structure:** variable rate (7.3% vs 4.0%), revolving lines (8.6% vs 5.9%) and no collateral (11.1% vs 5.0% with collateral) all raise risk.
- **Borrower:** established businesses (2+ years) are safer (6.0% vs 7.7%); an out-of-state lender doubles the rate (7.9% vs 4.0%).

**Recalibration:** isotonic regression fitted on FY2018 lowers the over-prediction (average predicted 8.3% -> 7.9%, actual 6.6%) without changing the ranking. Some over-prediction remains because FY2019 loans are younger; in use the calibration should be refreshed yearly.

**Risk tiers (FY2019)**

| Tier | Share of loans | Actual bad rate | Share of bad loans | Loss rate on $ approved |
|---|---|---|---|---|
| A (safest) | 42% | 2.0% | 13% | 0.8% |
| B | 30% | 5.6% | 25% | 2.7% |
| C | 14% | 9.9% | 20% | 5.2% |
| D | 9% | 14.7% | 21% | 8.7% |
| E (riskiest) | 6% | 25.5% | 21% | 20.3% |

Tiers D and E hold 15% of loans but 42% of bad loans; tier E loses 20% of the dollars approved, 25 times tier A.

**Cost-based review rule:** review a loan when its expected saving (risk x lender exposure x share of loss avoided) exceeds the review cost. With a $500 review that avoids 20% of the loss, the rule reviews 39% of FY2019 loans, covers 76% of the lender's losses and nets **$8.8m**, against $4.2m for reviewing the same number ranked by probability alone and $0.7m at random. The result depends on the assumptions (sensitivity table), which a lending team should replace with its own.

**Pricing view:** expected lender loss implies a risk premium of roughly 6 bp a year for tier A rising to about 86 bp for tier E (about 80 bp more), before funding, servicing and capital costs.